# Guía Completa de Python para Modelos y Sistemas (y APS)
*Traducción y equivalencia 1 a 1 de MATLAB a Python (NumPy, SciPy, Matplotlib y SciPy Signal)*

---

Esta guía es la traducción directa, práctica y exhaustiva de todos los conceptos del notebook de MATLAB (`IntroMATlab.ipynb`) y del **Resumen de MATLAB para Modelos y Sistemas**.

### ¿Qué herramientas se usan en Python para Modelos y Sistemas?
- **`NumPy`** (`import numpy as np`): Operaciones matriciales, raíces de polinomios, arreglos y números aleatorios.
- **`Matplotlib`** (`import matplotlib.pyplot as plt`): Gráficos 2D, diagramas de fase (`streamplot`), gráficos de tallo (`stem`), polares y diagramas de Bode.
- **`SciPy` / `SciPy Signal`** (`import scipy.signal as signal` / `from scipy.integrate import solve_ivp`):
  - Definición de funciones de transferencia (`signal.TransferFunction`) y espacio de estados (`signal.StateSpace`).
  - Conversión de modelos (`ss2tf`, `tf2ss`).
  - Respuestas temporales (`step`, `impulse`, `initial`, `lsim`).
  - Diagramas de Bode (`signal.bode`) y cálculo de ceros/polos (`signal.tf2zpk`).
  - Filtrado discreto (`lfilter`), convolución (`np.convolve`) y FFT (`np.fft.fft`).


## 1. Importación de Librerías Principales
En Python comenzamos importando las librerías necesarias con sus convenios estándar:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as signal
from scipy.integrate import solve_ivp
import sys


## 2. Comandos de Ayuda y Documentación

| MATLAB | Python / IPython (Colab) | Descripción |
| :--- | :--- | :--- |
| `help funcion` | `help(funcion)` o `funcion?` | Documentación oficial de la función |
| `doc funcion` | `help(funcion)` o `funcion??` | Documentación extendida / código fuente |
| `lookfor palabra` | `dir(modulo)` o búsqueda en docs | Búsqueda de métodos en una librería |
| `demo` | Notebooks de ejemplo / Docs oficial | Ejemplos de uso |


In [ ]:
# Ejemplo de consulta de ayuda en Python
help(signal.TransferFunction)


## 3. Variables, Reglas de Nombramiento y Formato de Salida

| MATLAB | Python / IPython (Colab) |
| :--- | :--- |
| `ans` | `_` (guión bajo en consola guarda la última salida) |
| `who` / `whos` | `dir()` o `%whos` en Colab/Jupyter |
| `clear x` | `del x` |
| `class(x)` | `type(x)` o `x.dtype` (para arrays NumPy) |
| `format long` / `format short` | `np.set_printoptions(precision=4)` o f-strings `f"{val:.4f}"` |
| Continuación `...` | Paréntesis `()` o barra invertida `\` |
| `clc` | Limpiar consola (en Jupyter: `from IPython.display import clear_output; clear_output()`) |


In [ ]:
# Creación de variables y tipos
a = 43
b = 3.141592653589793

# Tipo de variable (class)
print("Tipo de 'a':", type(a))

# Formateo de salida (format short vs long)
print(f"Format short (4 dec): {b:.4f}")
print(f"Format long (15 dec): {b:.15f}")

# Casteo de tipos (cast / uint16)
a_uint16 = np.uint16(43)
b_cast = np.uint16(b)
print("Casteado a uint16:", b_cast, type(b_cast))


## 4. Precisión Numérica y Números Aleatorios

- **Épsilon de máquina (`eps`)**: Precisión de coma flotante (`np.finfo(np.float64).eps`).
- **Aleatorios (`rand`, `randn`, `randi`, `rng`)**: Módulo `np.random`.


In [ ]:
# Épsilon de máquina (eps)
eps_val = np.finfo(np.float64).eps
print("Épsilon de máquina (eps):", eps_val)

# Fijar semilla aleatoria (rng(42))
np.random.seed(42)

# rand(n, m) -> Matriz uniforme entre 0 y 1
print("\nrand(3,3):\n", np.random.rand(3, 3))

# randn(n, m) -> Matriz distribución normal
print("\nrandn(3,3):\n", np.random.randn(3, 3))

# Aleatorios en intervalo [low, high] -> (rand*(high-low) + low)
print("\nUniforme [5, 15]:\n", np.random.uniform(5, 15, size=(2, 2)))

# Enteros aleatorios (randi / round(rand*10))
print("\nEnteros aleatorios [0, 10]:\n", np.random.randint(0, 11, size=(2, 3)))


## 5. Rango, Vectores, Zeros, Ones y Meshgrid

| Concepto | MATLAB | Python (NumPy) |
| :--- | :--- | :--- |
| Rango dos puntos | `1:2:10` *(inicio:paso:fin)* | `np.arange(1, 11, 2)` *(inicio, fin_exclusivo, paso)* |
| Vector linealmente espaciado | `linspace(x, y, n)` | `np.linspace(x, y, n)` |
| Matriz de ceros | `zeros(filas, cols)` | `np.zeros((filas, cols))` |
| Matriz de unos | `ones(filas, cols)` | `np.ones((filas, cols))` |
| Malla 2D | `[X, Y] = meshgrid(x, y)` | `X, Y = np.meshgrid(x, y)` |


In [ ]:
# arange y linspace
print("arange:", np.arange(1, 10, 2))
print("linspace:", np.linspace(0, 2*np.pi, 5))

# Zeros y Ones
print("Zeros (2x3):\n", np.zeros((2, 3)))
print("Ones (2x3):\n", np.ones((2, 3)))

# Meshgrid 2D
x_vec = np.linspace(-1, 1, 3)
y_vec = np.linspace(-1, 1, 3)
X, Y = np.meshgrid(x_vec, y_vec)
print("\nMeshgrid X:\n", X)
print("Meshgrid Y:\n", Y)


## 6. Indexación, Transposición y Manipulación de Matrices

> ⚠️ **Atención:** En MATLAB la indexación empieza en `1`. En Python en `0`.

- **Último elemento:** En MATLAB es `end`. En Python es `-1`.
- **Transpuesta:** En MATLAB es `mat'`. En Python es `mat.T`.
- **Girar matriz:** `fliplr(mat)` ➔ `np.fliplr(mat)`, `flipud(mat)` ➔ `np.flipud(mat)`.
- **Cambiar forma:** `reshape(mat, 2, 6)` ➔ `np.reshape(mat, (2, 6))`.
- **Eliminar elementos (`v(1) = []`):** `v = np.delete(v, 0)`.


In [ ]:
mat = np.array([
    [1, 2, 3],
    [44, 9, 2],
    [5, 4, 3]
])

print("Elemento fila 3, col 2 (MATLAB: mat(3,2)):", mat[2, 1])
print("Primera fila (mat(1,:)):", mat[0, :])
print("Segunda columna (mat(:,2)):", mat[:, 1])

# Transpuesta
print("Transpuesta (mat.T):\n", mat.T)

# Girar matrices
print("Fliplr:\n", np.fliplr(mat))
print("Flipud:\n", np.flipud(mat))

# Reshape
mat_ampliada = np.hstack((mat, np.array([[8], [11], [33]])))
print("Reshape a 2x6:\n", mat_ampliada.reshape(2, 6))

# Borrar elemento (v(1) = [])
v = np.array([10, 20, 30, 40])
v = np.delete(v, 0)
print("Vector tras borrar primer elemento:", v)


## 7. Funciones Numéricas, Búsquedas y Comparaciones

| Función | MATLAB | Python (NumPy) |
| :--- | :--- | :--- |
| Dimensiones | `size(mat)` | `mat.shape` |
| Número de elementos | `numel(mat)` | `mat.size` |
| Largo de vector | `length(v)` | `len(v)` |
| Valor absoluto | `abs(x)` | `np.abs(x)` |
| Mínimo y Máximo | `min(x)`, `max(x)` | `np.min(x)`, `np.max(x)` |
| Algún / Todos | `any(x)`, `all(x)` | `np.any(x)`, `np.all(x)` |
| Buscar índices | `find(x > 5)` | `np.where(x > 5)[0]` |
| Comparar matrices | `isequal(A, B)` | `np.array_equal(A, B)` |
| Raíces de polinomio | `roots([1 2 0 1])` | `np.roots([1, 2, 0, 1])` |
| Convolución | `conv(h, x)` | `np.convolve(h, x)` |
| Transformada Fourier | `fft(x)` | `np.fft.fft(x)` |


In [ ]:
arr = np.array([10, -5, 20, 0, 15])

print("Shape:", arr.shape, "| Size:", arr.size, "| Len:", len(arr))
print("Absoluto:", np.abs(arr))
print("Min y Max:", np.min(arr), np.max(arr))

# Búsqueda (find)
print("Índices arr > 5:", np.where(arr > 5)[0])

# Raíces de un polinomio p(x) = x^3 + 2x^2 + 1  (MATLAB: roots([1 2 0 1]))
poly_roots = np.roots([1, 2, 0, 1])
print("Raíces del polinomio x^3 + 2x^2 + 1:", poly_roots)

# Convolución de vectores (MATLAB: conv(h, x))
h = np.array([0.5, 1.0, 0.5])
x_signal = np.array([1, 2, 3, 0, -1])
conv_res = np.convolve(h, x_signal)
print("Convolución h * x:", conv_res)

# Transformada Rápida de Fourier (MATLAB: fft(x))
fft_res = np.fft.fft(x_signal)
print("FFT de la señal:", fft_res)


## 8. Lectura y Escritura de Archivos (save / load)

En MATLAB: `save testfile.dat mat2 -ascii` y `load testfile.dat`.
En Python: `np.savetxt` y `np.loadtxt`.


In [ ]:
datos = np.array([[1, 2, 3], [4, 5, 6]])

# Guardar en ascii (save -ascii)
np.savetxt("testfile.dat", datos, fmt="%.4f")

# Cargar desde archivo (load)
datos_cargados = np.loadtxt("testfile.dat")
print("Datos cargados:\n", datos_cargados)


## 9. Gráficos y Visualización (`matplotlib.pyplot`)

En MATLAB: `figure`, `bar`, `plot`, `stem`, `polarplot`, `subplot`, `hold on`, `grid on`, `legend`, `xlabel`, `ylabel`, `title`.

En Python:
- No requiere `hold on` (los trazados se superponen automáticamente).
- Gráficos discretos de impulsos/tallo: `plt.stem()`.
- Gráficos polares: `plt.polar()`.


In [ ]:
x = np.linspace(0, 2*np.pi, 30)
y_sin = np.sin(x)
y_cos = np.cos(x)

# Subplots (equivalente a subplot(1, 2, 1) y subplot(1, 2, 2))
plt.figure(figsize=(12, 4))

# Subplot 1: Gráfico de líneas compuestas (plot / hold on)
plt.subplot(1, 2, 1)
plt.plot(x, y_sin, 'ro-', label='sin(x)')
plt.plot(x, y_cos, 'b+--', label='cos(x)')
plt.grid(True)
plt.legend()
plt.title('Gráfico de Líneas Continuas')
plt.xlabel('x')
plt.ylabel('y')

# Subplot 2: Gráfico Discreto en Tallo (MATLAB: stem)
plt.subplot(1, 2, 2)
plt.stem(x, y_sin, linefmt='b-', markerfmt='bo', basefmt='r-')
plt.grid(True)
plt.title('Gráfico Discreto en Tallo (stem)')
plt.xlabel('n')
plt.ylabel('y[n]')

plt.tight_layout()
plt.show()


## 10. Definición de Funciones en Python

En MATLAB: `function [y1, y2] = mi_funcion(x1, x2) ... end`.
En Python: `def mi_funcion(x1, x2): ... return y1, y2`.


In [ ]:
def mi_funcion(x1, x2):
    """Docstring: Devuelve suma y producto."""
    return x1 + x2, x1 * x2

suma, prod = mi_funcion(5, 3)
print(f"Suma: {suma}, Producto: {prod}")


## 11. Solucionadores de Ecuaciones Diferenciales Ordinarias (ODE Solvers)

### Tabla de Equivalencias MATLAB vs SciPy (`solve_ivp`):

| Método MATLAB | Algoritmo y Tipo de Problema | Equivalente en SciPy (`solve_ivp`) |
| :--- | :--- | :--- |
| **`ode45`** | Runge-Kutta (4,5) explícito (Dormand-Prince). Para sistemas no rígidos. | `method='RK45'` *(por defecto)* |
| **`ode23`** | Runge-Kutta (2,3) explícito (Bogacki-Shampine). | `method='RK23'` |
| **`ode113`** | Adams-Bashforth-Moulton de orden variable. | `method='LSODA'` |
| **`ode15s`** | NDF / BDF (Gear's method) para **sistemas rígidos (*stiff*)** o DAE. | `method='BDF'` o `method='Radau'` |
| **`ode23s`** | Rosenbrock modificado de orden 2 para problemas rígidos. | `method='Radau'` |
| **`ode23t`** / **`ode23tb`** | Regla trapezoidal / TR-BDF2 sin amortiguamiento numérico. | `method='Radau'` o `method='BDF'` |


In [ ]:
def sistema_amortiguado(t, y, wn=2.0, zeta=0.25):
    y1, y2 = y
    dy1_dt = y2
    dy2_dt = -2 * zeta * wn * y2 - (wn**2) * y1
    return [dy1_dt, dy2_dt]

y0 = [1.0, 0.0]
t_span = (0, 10)
t_eval = np.linspace(0, 10, 200)

sol_rk45 = solve_ivp(sistema_amortiguado, t_span, y0, method='RK45', t_eval=t_eval)

plt.figure(figsize=(8, 4))
plt.plot(sol_rk45.t, sol_rk45.y[0], 'b-', label='Posición y(t) [ode45/RK45]')
plt.plot(sol_rk45.t, sol_rk45.y[1], 'r--', label="Velocidad y'(t) [ode45/RK45]")
plt.grid(True)
plt.xlabel('Tiempo t')
plt.title('Simulación de EDO en Python')
plt.legend()
plt.show()


## 12. Funciones Específicas de Modelos y Sistemas (`scipy.signal`)

En la materia **Modelos y Sistemas** se utilizan ampliamente funciones de **Sistemas Lineales e Invariantes en el Tiempo (LTI)**, Funciones de Transferencia, Espacio de Estados, Diagramas de Bode, Polos y Ceros, Respuestas al Escalón e Impulso y Diagramas de Fase.

Aquí está la equivalencia exacta de cada comando de MATLAB en Python:

| Comando MATLAB | Descripción | Equivalente en Python (`scipy.signal` / `matplotlib`) |
| :--- | :--- | :--- |
| `H = tf(num, den)` | Crea la función de transferencia $\frac{Num(s)}{Den(s)}$ | `sys = signal.TransferFunction(num, den)` |
| `sys = ss(A, B, C, D)` | Crea la representación en espacio de estados | `sys = signal.StateSpace(A, B, C, D)` |
| `[num, den] = ss2tf(A, B, C, D)` | Convierte espacio de estados a función transferencia | `num, den = signal.ss2tf(A, B, C, D)` |
| `[A, B, C, D] = tf2ss(num, den)` | Convierte función transferencia a espacio de estados | `A, B, C, D = signal.tf2ss(num, den)` |
| `[y, t] = step(sys)` | Respuesta al escalón unitario | `t, y = signal.step(sys)` |
| `[y, t] = impulse(sys)` | Respuesta al impulso unitario | `t, y = signal.impulse(sys)` |
| `[y, t] = initial(sys, x0)` | Respuesta a condiciones iniciales | `t, y, x = signal.initial(sys, X0=x0, return_x=True)` |
| `y = lsim(sys, u, t)` | Respuesta a cualquier entrada arbitraria $u(t)$ | `t, y, x = signal.lsim(sys, U=u, T=t)` |
| `[p, z] = pzmap(sys)` / `zplane` | Polos y Ceros del sistema | `z, p, k = signal.tf2zpk(num, den)` |
| `bode(sys)` | Diagrama de Bode (Magnitud en dB y Fase en °) | `w, mag, phase = signal.bode(sys)` |
| `y = filter(b, a, x)` | Filtrado de señal discreta | `y = signal.lfilter(b, a, x)` |
| `plot(x(:,1), x(:,2))` | Diagrama de Fases (Espacio de Estados 2D) | `plt.plot(x1, x2)` y `plt.streamplot()` |

---

### Ejemplo Práctico 1: Función de Transferencia $H(s)$, Espacio de Estados, Escalón e Impulso
Dada la función de transferencia $H(s) = \frac{2s + 1}{s^2 + 4s + 5}$:


In [ ]:
# 1. Definir numerador y denominador H(s) = (2s + 1) / (s^2 + 4s + 5)
num = [2, 1]
den = [1, 4, 5]

# Crear objeto de Función de Transferencia (MATLAB: H = tf(num, den))
H_tf = signal.TransferFunction(num, den)
print("Función de Transferencia H(s):\n", H_tf)

# Conversión a Espacio de Estados (MATLAB: [A, B, C, D] = tf2ss(num, den))
A, B, C, D = signal.tf2ss(num, den)
sys_ss = signal.StateSpace(A, B, C, D)
print("\nMatriz A del Espacio de Estados:\n", A)

# Respuesta al Escalón (MATLAB: step(H))
t_step, y_step = signal.step(H_tf)

# Respuesta al Impulso (MATLAB: impulse(H))
t_imp, y_imp = signal.impulse(H_tf)

# Graficar ambas respuestas
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(t_step, y_step, 'b-', linewidth=2)
plt.title('Respuesta al Escalón (step)')
plt.xlabel('Tiempo [s]')
plt.ylabel('y(t)')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(t_imp, y_imp, 'r-', linewidth=2)
plt.title('Respuesta al Impulso (impulse)')
plt.xlabel('Tiempo [s]')
plt.ylabel('y(t)')
plt.grid(True)

plt.tight_layout()
plt.show()


### Ejemplo Práctico 2: Diagrama de Bode, Polos y Ceros (`bode` y `pzmap`)


In [ ]:
# Polos y Ceros (MATLAB: [p, z] = pzmap(H))
zeros_pts, poles_pts, _ = signal.tf2zpk(num, den)
print("Ceros del sistema:", zeros_pts)
print("Polos del sistema:", poles_pts)

# Diagrama de Polos y Ceros (PZMap)
plt.figure(figsize=(5, 5))
plt.scatter(np.real(poles_pts), np.imag(poles_pts), marker='x', color='red', s=100, label='Polos (x)')
plt.scatter(np.real(zeros_pts), np.imag(zeros_pts), marker='o', color='blue', s=100, facecolors='none', label='Ceros (o)')
plt.axhline(0, color='black', linewidth=0.8, linestyle='--')
plt.axvline(0, color='black', linewidth=0.8, linestyle='--')
plt.grid(True)
plt.xlabel('Eje Real (Sigma)')
plt.ylabel('Eje Imaginario (j W)')
plt.title('Mapa de Polos y Ceros (PZMap)')
plt.legend()
plt.show()

# Diagrama de Bode (MATLAB: bode(H))
w, mag, phase = signal.bode(H_tf)

plt.figure(figsize=(10, 5))

# Magnitud en dB
plt.subplot(2, 1, 1)
plt.semilogx(w, mag, 'b-', linewidth=2)
plt.title('Diagrama de Bode - Magnitud y Fase')
plt.ylabel('Magnitud [dB]')
plt.grid(True, which='both')

# Fase en grados
plt.subplot(2, 1, 2)
plt.semilogx(w, phase, 'r-', linewidth=2)
plt.xlabel('Frecuencia [rad/s]')
plt.ylabel('Fase [grados]')
plt.grid(True, which='both')

plt.tight_layout()
plt.show()


### Ejemplo Práctico 3: Diagrama de Fases (Retrato de Fases 2D en Espacio de Estados)

En el estudio de estabilidad de sistemas dinámicos (Nodos, Sillas, Focos, Centros), graficamos las trayectorias en el plano de estados $(x_1, x_2)$ y el campo de vectores.


In [ ]:
# Sistema lineal de 2do orden: dx1/dt = x2, dx2/dt = -2*x1 - 3*x2 (Nodo Estable)
x1_vals = np.linspace(-3, 3, 20)
x2_vals = np.linspace(-3, 3, 20)
X1, X2 = np.meshgrid(x1_vals, x2_vals)

dX1_dt = X2
dX2_dt = -2*X1 - 3*X2

plt.figure(figsize=(7, 6))

# Campo de vectores de velocidad (streamplot)
plt.streamplot(X1, X2, dX1_dt, dX2_dt, color='blue', density=1.2)
plt.plot(0, 0, 'ro', markersize=8, label='Punto de Equilibrio (Origen)')
plt.grid(True)
plt.xlabel('Estado x1(t)')
plt.ylabel('Estado x2(t)')
plt.title('Diagrama / Retrato de Fases en Espacio de Estados')
plt.legend()
plt.show()


---
### 📌 Resumen para la cursada de Modelos y Sistemas con Python:
1. **LTI y Transferencia:** Usa `scipy.signal.TransferFunction` y `scipy.signal.StateSpace`.
2. **Simulaciones:** `signal.step`, `signal.impulse`, `signal.initial`, `signal.lsim`.
3. **Estabilidad y Frecuencia:** `signal.tf2zpk` para ceros/polos, `signal.bode` para Bode, y `plt.streamplot` para Diagramas de Fase.
